# Google Search Ranking & Discoverability Capstone
## CTR Opportunity Scoring with a Future-Window, Client-Holdout Evaluation

**Lane:** CTR / Engagement Opportunity Scoring  
**Research question:** Can observable search signals from the prior 60 days rank content items that will under-capture clicks in the following 30 days, compared with a transparent position-adjusted rule?  
**Decision supported:** Prioritise a small review queue for a human to inspect. This is not an automatic metadata-change system and does not estimate causal lift.

**Data:** FlyRank/internship-warehouse, build `flyrank_pseudonymized_warehouse_release_v20260703`.  
**Feature window:** 2026-03-02 through 2026-04-30.  
**Outcome window:** 2026-05-01 through 2026-05-31.  
**Unit:** client-content pair.  
**Validation:** grouped holdout by client, so no client appears in both train and test.

> Run top to bottom in Colab. Add your Hugging Face read token as a Colab Secret named `HF_TOKEN`. Never paste a token into a cell or commit it. This notebook reads the gated warehouse directly with DuckDB; it does not download the 79M-row fact table into pandas.

## 1. Setup and data contract

We use only `fact_content_daily_performance`. We deliberately exclude query-level data because its fixed 90-day window does not align cleanly with this decision point. We also exclude GA4 engagement fields to keep the capstone focused on search CTR and avoid mixing measurement-availability regimes.

The feature window ends on 2026-04-30. The target window begins on 2026-05-01, so the windows do not overlap. IDs are used only for grouping and client-holdout splitting, never as model features. The public paper receives aggregate metrics only; the ranked CSV stays local and is not committed.

In [ ]:
%pip -q install duckdb huggingface_hub pandas numpy scikit-learn matplotlib

In [ ]:
import os, json, math, warnings, zipfile, subprocess
from pathlib import Path

# Colab opens the notebook file, not a working checkout of the repository.
REPO_URL = "https://github.com/mhdaamiryo/fly-ml.git"
REPO_ROOT = Path("/content/fly-ml")
if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)
os.chdir(REPO_ROOT)
print("Working repository:", Path.cwd())
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, f1_score
from sklearn.inspection import permutation_importance

SEED = 42
np.random.seed(SEED)

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Add your Hugging Face READ token in Colab Secrets as HF_TOKEN.")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"

FEATURE_START = "2026-03-02"
FEATURE_MID = "2026-04-01"
FEATURE_END = "2026-05-01"   # exclusive
TARGET_START = "2026-05-01"
TARGET_END = "2026-06-01"    # exclusive

print("Warehouse connection ready.")
print("Feature window:", FEATURE_START, "to", "2026-04-30")
print("Target window:", TARGET_START, "to", "2026-05-31")

## 2. Build one row per client-content pair in DuckDB

DuckDB aggregates the remote Parquet data before pandas receives anything. The two halves of the feature window let us calculate a simple momentum feature without looking into the target period. The outcome uses future CTR only after the feature frame is complete.

In [ ]:
sql = f"""
WITH agg AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(CASE WHEN report_date >= DATE '{FEATURE_START}' AND report_date < DATE '{FEATURE_MID}'
                 THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS impressions_older30,
        SUM(CASE WHEN report_date >= DATE '{FEATURE_START}' AND report_date < DATE '{FEATURE_MID}'
                 THEN COALESCE(gsc_clicks, 0) ELSE 0 END) AS clicks_older30,

        SUM(CASE WHEN report_date >= DATE '{FEATURE_MID}' AND report_date < DATE '{FEATURE_END}'
                 THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS impressions_recent30,
        SUM(CASE WHEN report_date >= DATE '{FEATURE_MID}' AND report_date < DATE '{FEATURE_END}'
                 THEN COALESCE(gsc_clicks, 0) ELSE 0 END) AS clicks_recent30,

        AVG(CASE WHEN report_date >= DATE '{FEATURE_START}' AND report_date < DATE '{FEATURE_END}'
                 THEN NULLIF(gsc_avg_position, 0) END) AS prior_avg_position,
        STDDEV_SAMP(CASE WHEN report_date >= DATE '{FEATURE_START}' AND report_date < DATE '{FEATURE_END}'
                         THEN NULLIF(gsc_avg_position, 0) END) AS prior_position_sd,
        COUNT(DISTINCT CASE WHEN report_date >= DATE '{FEATURE_START}' AND report_date < DATE '{FEATURE_END}'
                             AND COALESCE(gsc_impressions, 0) > 0 THEN report_date END) AS prior_days_with_impressions,

        SUM(CASE WHEN report_date >= DATE '{TARGET_START}' AND report_date < DATE '{TARGET_END}'
                 THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS future_impressions,
        SUM(CASE WHEN report_date >= DATE '{TARGET_START}' AND report_date < DATE '{TARGET_END}'
                 THEN COALESCE(gsc_clicks, 0) ELSE 0 END) AS future_clicks,
        COUNT(DISTINCT CASE WHEN report_date >= DATE '{TARGET_START}' AND report_date < DATE '{TARGET_END}'
                             AND COALESCE(gsc_impressions, 0) > 0 THEN report_date END) AS future_days_with_impressions
    FROM {FACT}
    WHERE report_date >= DATE '{FEATURE_START}'
      AND report_date < DATE '{TARGET_END}'
    GROUP BY client_hash_id, content_hash_id
)
SELECT *
FROM agg
WHERE impressions_older30 + impressions_recent30 >= 500
  AND future_impressions >= 100
"""
data = con.sql(sql).df()

numeric_cols = [c for c in data.columns if c not in ["client_hash_id", "content_hash_id"]]
for c in numeric_cols:
    data[c] = pd.to_numeric(data[c], errors="coerce")

data["prior_impressions"] = data["impressions_older30"] + data["impressions_recent30"]
data["prior_clicks"] = data["clicks_older30"] + data["clicks_recent30"]
data["prior_ctr_pct"] = np.where(data["prior_impressions"] > 0, 100 * data["prior_clicks"] / data["prior_impressions"], np.nan)
data["recent_ctr_pct"] = np.where(data["impressions_recent30"] > 0, 100 * data["clicks_recent30"] / data["impressions_recent30"], np.nan)
data["older_ctr_pct"] = np.where(data["impressions_older30"] > 0, 100 * data["clicks_older30"] / data["impressions_older30"], np.nan)
data["future_ctr_pct"] = np.where(data["future_impressions"] > 0, 100 * data["future_clicks"] / data["future_impressions"], np.nan)
data["impression_change_pct"] = np.where(
    data["impressions_older30"] > 0,
    100 * (data["impressions_recent30"] - data["impressions_older30"]) / data["impressions_older30"],
    np.nan
)
data["position_band"] = pd.cut(
    data["prior_avg_position"],
    bins=[0, 3, 10, 20, 50, np.inf],
    labels=["1-3", "4-10", "11-20", "21-50", "50+"],
    include_lowest=True
)

print(f"Eligible client-content pairs: {len(data):,}")
print(f"Clients represented: {data['client_hash_id'].nunique():,}")
print(f"Rows with prior position: {data['prior_avg_position'].notna().sum():,}")
print(f"Total future impressions: {data['future_impressions'].sum():,.0f}")
assert len(data) > 100, "Unexpectedly small feature frame; inspect date filters and warehouse access."
assert data["client_hash_id"].nunique() >= 5, "Too few client groups for grouped validation."

## 3. Define the future outcome without leakage

The target is a **future under-capture flag**. Within each prior-position band, calculate the 25th percentile of *positive* future CTR on training clients only. A row is positive when its future CTR is below that training-derived threshold. Positive-only benchmarking avoids a zero-valued lower quartile making the rule unusable when many pages receive no clicks.

The target window has a 100-impression minimum. The model sees only prior-window features. The client and content hashes are not features. Test labels use the thresholds learned from training clients; the test set does not set its own benchmark.

In [ ]:
FEATURES = [
    "prior_impressions", "prior_clicks", "prior_ctr_pct",
    "recent_ctr_pct", "older_ctr_pct", "impression_change_pct",
    "prior_avg_position", "prior_position_sd",
    "prior_days_with_impressions", "future_days_with_impressions"
]
# future_days_with_impressions is an outcome-window variable, so explicitly remove it.
FEATURES = [c for c in FEATURES if c != "future_days_with_impressions"]

eligible = data.dropna(subset=["position_band", "future_ctr_pct"]).copy()
groups = eligible["client_hash_id"].astype(str)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
train_idx, test_idx = next(splitter.split(eligible[FEATURES], groups=groups))
train = eligible.iloc[train_idx].copy()
test = eligible.iloc[test_idx].copy()

# Learn the future-CTR reference threshold using positive future CTR on TRAIN clients only.
future_thresholds = (
    train.loc[train["future_ctr_pct"] > 0]
         .groupby("position_band", observed=False)["future_ctr_pct"]
         .quantile(0.25)
)
train["future_ctr_reference"] = train["position_band"].map(future_thresholds).astype(float)
test["future_ctr_reference"] = test["position_band"].map(future_thresholds).astype(float)

train = train.dropna(subset=["future_ctr_reference"]).copy()
test = test.dropna(subset=["future_ctr_reference"]).copy()
train["under_capture"] = (train["future_ctr_pct"] < train["future_ctr_reference"]).astype(int)
test["under_capture"] = (test["future_ctr_pct"] < test["future_ctr_reference"]).astype(int)

print("Position-band future CTR thresholds learned on training clients:")
display(future_thresholds.rename("future_ctr_p25_positive").to_frame())
print(f"Train rows: {len(train):,} | clients: {train['client_hash_id'].nunique():,} | positive rate: {train['under_capture'].mean():.3f}")
print(f"Test rows:  {len(test):,} | clients: {test['client_hash_id'].nunique():,} | positive rate: {test['under_capture'].mean():.3f}")
assert set(train["client_hash_id"]).isdisjoint(set(test["client_hash_id"])), "Client leakage across train/test."
assert train["under_capture"].nunique() == 2 and test["under_capture"].nunique() == 2, "Grouped split has only one target class; inspect the split and data coverage."

# Leakage guard: target/outcome fields are not model inputs.
FORBIDDEN = {"future_impressions", "future_clicks", "future_ctr_pct", "future_ctr_reference", "under_capture",
             "client_hash_id", "content_hash_id", "future_days_with_impressions"}
assert not (set(FEATURES) & FORBIDDEN)
print("Leakage guard passed: features are prior-window measurements only; IDs are grouping-only.")

## 4. Transparent position-adjusted baseline

The baseline ranks pages by prior-window CTR gap against the 25th percentile of positive prior CTR in the same position band, multiplied by prior impressions. This is intentionally simple and available before the future window. It is evaluated against the same future target and same client-held-out test rows as the learned models.

In [ ]:
prior_thresholds = (
    train.loc[train["prior_ctr_pct"] > 0]
         .groupby("position_band", observed=False)["prior_ctr_pct"]
         .quantile(0.25)
)
for frame in (train, test):
    frame["prior_ctr_reference"] = frame["position_band"].map(prior_thresholds).astype(float)
    frame["baseline_gap_pp"] = (frame["prior_ctr_reference"] - frame["prior_ctr_pct"]).clip(lower=0)
    frame["baseline_score"] = frame["prior_impressions"] * frame["baseline_gap_pp"] / 100

print("Baseline eligible test rows:", int((test["baseline_score"] > 0).sum()))
display(prior_thresholds.rename("prior_ctr_p25_positive").to_frame())

## 5. Train models and compare on the same split

We compare the transparent baseline, Logistic Regression and Random Forest. The models use the same feature set, training rows and held-out test rows. The primary ranking metrics are average precision and Precision@50; ROC AUC, recall and F1 are secondary. The test positive rate is reported beside them so a reader can judge the base rate.

In [ ]:
X_train, y_train = train[FEATURES], train["under_capture"]
X_test, y_test = test[FEATURES], test["under_capture"]

models = {
    "logistic_regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=1500, class_weight="balanced", random_state=SEED))
    ]),
    "random_forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=350, min_samples_leaf=10,
            class_weight="balanced_subsample", random_state=SEED, n_jobs=-1
        ))
    ])
}

def precision_at_k(y_true, scores, k=50):
    k = min(k, len(y_true))
    order = np.argsort(np.asarray(scores))[::-1][:k]
    return float(np.asarray(y_true)[order].mean()) if k else float("nan")

def evaluate_scores(name, y_true, scores, threshold=0.5):
    pred = (np.asarray(scores) >= threshold).astype(int)
    return {
        "method": name,
        "average_precision": float(average_precision_score(y_true, scores)),
        "roc_auc": float(roc_auc_score(y_true, scores)),
        "precision_at_50": precision_at_k(y_true, scores, 50),
        "recall_at_0_5": float(recall_score(y_true, pred, zero_division=0)),
        "f1_at_0_5": float(f1_score(y_true, pred, zero_division=0))
    }

# The primary model is chosen a priori for interpretability, not selected by test performance.
# Random Forest is retained as a non-linear comparator. Both are evaluated on the same untouched test set.
PRIMARY_MODEL = "logistic_regression"
fitted_models = {}
model_scores = {}
results = [evaluate_scores(
    "position_adjusted_baseline", y_test,
    test["baseline_score"].fillna(0).to_numpy()
)]

for name, model in models.items():
    model.fit(X_train, y_train)
    scores = model.predict_proba(X_test)[:, 1]
    fitted_models[name] = model
    model_scores[name] = scores
    results.append(evaluate_scores(name, y_test, scores))

metrics = pd.DataFrame(results)
print(f"Test base rate: {y_test.mean():.3f} ({int(y_test.sum()):,}/{len(y_test):,})")
display(metrics.round(4))
best_model_name = PRIMARY_MODEL
print("Primary model fixed before test evaluation:", best_model_name)
print("Random Forest is reported as a comparator; the test set is not used to select the primary model.")

## 6. Error analysis and interpretation

A ranking metric is not enough. Inspect false positives and false negatives at the 0.50 probability threshold, then examine permutation importance on held-out clients. Importance is predictive association, not a causal explanation. Rows printed below contain only aggregate metrics and no IDs.

In [ ]:
best_model = fitted_models[best_model_name]
best_scores = model_scores[best_model_name]
best_pred = (best_scores >= 0.5).astype(int)
error_frame = test[["prior_impressions", "prior_ctr_pct", "prior_avg_position", "future_ctr_pct", "under_capture"]].copy()
error_frame["predicted_probability"] = best_scores
error_frame["prediction"] = best_pred
error_frame["error_type"] = np.select(
    [(error_frame["prediction"] == 1) & (error_frame["under_capture"] == 0),
     (error_frame["prediction"] == 0) & (error_frame["under_capture"] == 1)],
    ["false_positive", "false_negative"], default="correct"
)
error_summary = error_frame.groupby("error_type", observed=False).agg(
    n=("under_capture", "size"),
    mean_prior_impressions=("prior_impressions", "mean"),
    median_prior_position=("prior_avg_position", "median"),
    mean_future_ctr=("future_ctr_pct", "mean")
).reset_index()
display(error_summary.round(3))

perm = permutation_importance(
    best_model, X_test, y_test, scoring="average_precision",
    n_repeats=5, random_state=SEED, n_jobs=-1
)
importance = pd.DataFrame({
    "feature": FEATURES,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)
display(importance.head(10).round(4))

## 7. Ranked recommendations

The queue is a reviewer aid. Each row receives one action and one reason code. The top-50 model queue is exported locally with pseudonymous IDs so the authorised owner can map records inside approved systems. Do not publish the row-level queue. The public paper receives only aggregate counts and model metrics.

In [ ]:
# Score every eligible row with the selected model.
all_X = eligible[FEATURES]
all_scores = best_model.predict_proba(all_X)[:, 1]
queue = eligible[["client_hash_id", "content_hash_id", "prior_impressions",
                  "prior_ctr_pct", "prior_avg_position", "position_band"]].copy()
queue["model_probability"] = all_scores
queue["baseline_score"] = (
    queue["prior_impressions"] *
    (queue["position_band"].map(prior_thresholds).astype(float) - queue["prior_ctr_pct"]).clip(lower=0) / 100
)
queue["reason_code"] = np.where(
    queue["model_probability"] >= 0.5,
    "model_flagged_future_under_capture_risk", "monitor"
)
queue["action"] = np.where(
    queue["model_probability"] >= 0.5,
    "review_snippet_and_intent", "monitor"
)
queue = queue.sort_values(["model_probability", "prior_impressions"], ascending=[False, False]).reset_index(drop=True)
queue.insert(0, "rank", np.arange(1, len(queue) + 1))

# Local working queue includes hashes for authorised internal reconciliation; do not commit it.
out_dir = Path("work/outputs")
out_dir.mkdir(parents=True, exist_ok=True)
queue_path = out_dir / "capstone_ranked_queue.csv"
queue.to_csv(queue_path, index=False)
print("Local ranked queue:", queue_path, "| rows:", f"{len(queue):,}")
display(queue.head(10)[["rank", "model_probability", "reason_code", "action",
                        "prior_impressions", "prior_ctr_pct", "prior_avg_position"]].round(4))

## 8. Public-safe results and charts

This cell writes aggregate metrics and charts only. No client/content IDs, URLs, domains, titles, queries, tokens or row-level data enter the public assets. The JSON and SVG files are intended for the paper page in `docs/`.

In [ ]:
public_dir = Path("docs/assets")
public_dir.mkdir(parents=True, exist_ok=True)

# Model comparison chart
fig, ax = plt.subplots(figsize=(8, 4.5))
plot_metrics = metrics.set_index("method")[["average_precision", "precision_at_50"]]
plot_metrics.plot(kind="bar", ax=ax)
ax.set_title("Held-out ranking performance")
ax.set_ylabel("Score")
ax.set_xlabel("")
ax.set_ylim(0, 1)
ax.legend(["Average precision", "Precision@50"], frameon=False)
ax.tick_params(axis="x", rotation=15)
fig.tight_layout()
fig.savefig(public_dir / "model-comparison.svg", format="svg")
plt.show()

# Position-band future outcome chart
band_summary = test.groupby("position_band", observed=False).agg(
    n=("future_ctr_pct", "size"),
    median_future_ctr=("future_ctr_pct", "median")
).reset_index()
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(band_summary["position_band"].astype(str), band_summary["median_future_ctr"])
ax.set_title("Median future CTR by prior position band")
ax.set_xlabel("Prior average position")
ax.set_ylabel("Median future CTR (%)")
fig.tight_layout()
fig.savefig(public_dir / "future-ctr-by-position.svg", format="svg")
plt.show()

# Public-safe metrics payload; all values are aggregates.
public_payload = {
    "execution_required": False,
    "lane": "CTR / Engagement Opportunity Scoring",
    "release": "flyrank_pseudonymized_warehouse_release_v20260703",
    "feature_window": [FEATURE_START, "2026-04-30"],
    "outcome_window": [TARGET_START, "2026-05-31"],
    "eligible_rows": int(len(eligible)),
    "train_rows": int(len(train)),
    "test_rows": int(len(test)),
    "train_clients": int(train["client_hash_id"].nunique()),
    "test_clients": int(test["client_hash_id"].nunique()),
    "test_positive_rate": float(y_test.mean()),
    "selected_model": best_model_name,
    "metrics": results,
    "position_band_summary": [
        {"band": str(row.position_band), "n": int(row.n), "median_future_ctr": float(row.median_future_ctr)}
        for row in band_summary.itertuples(index=False)
    ],
    "top_features": [
        {"feature": str(row.feature), "importance": float(row.importance_mean)}
        for row in importance.head(8).itertuples(index=False)
    ],
    "queue_rows": int(len(queue)),
    "top50_action_count": int((queue.head(50)["action"] == "review_snippet_and_intent").sum()),
    "seed": SEED
}
(public_dir / "capstone_results.json").write_text(json.dumps(public_payload, indent=2))
print("Public-safe paper assets written to docs/assets/.")
print(json.dumps({k: public_payload[k] for k in ["eligible_rows","train_rows","test_rows","test_positive_rate","selected_model","top50_action_count"]}, indent=2))

## 9. Reproducibility, privacy and final checks

- Random seed: 42.
- Split: one 75/25 GroupShuffleSplit by client hash.
- Model features: prior impressions, prior clicks, prior CTR, two 30-day CTRs, impression change, prior average position, prior position variability, and days with impressions.
- Excluded: all future-window metrics, identifiers as features, GA4 fields, query-level fixed-window fields, raw text, URLs, domains and client names.
- Main caveat: one fixed feature/outcome window and one client holdout are a first capstone evaluation, not proof of stability across seasons or future releases.
- The future CTR threshold is learned from positive-CTR training rows by prior-position band. The test set does not define its own threshold.
- Recommendations are for human review only. Low CTR may reflect query intent, brand mix, SERP features, seasonality or measurement noise; do not assume metadata edits will cause improvement.

In [ ]:
# Final automated checks
assert FEATURE_END == TARGET_START, "Feature and target windows must be adjacent, not overlapping."
assert set(train["client_hash_id"]).isdisjoint(set(test["client_hash_id"]))
assert not (set(FEATURES) & FORBIDDEN)
assert len(metrics) == 3
assert (queue["rank"].to_numpy() == np.arange(1, len(queue) + 1)).all()
assert not any(c in public_payload for c in ["client_hash_id", "content_hash_id", "url", "query"])
assert all("client_hash_id" not in str(x) and "content_hash_id" not in str(x) for x in public_payload.keys())
print("All capstone checks passed.")
# Bundle only aggregate public assets so they can be uploaded without exposing the local ranked queue.
bundle_path = out_dir / "capstone_public_assets.zip"
with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for asset in sorted(public_dir.iterdir()):
        if asset.is_file():
            zf.write(asset, arcname=f"docs/assets/{asset.name}")
print("Public asset bundle:", bundle_path)
print("IMPORTANT: Save the executed notebook to GitHub. Upload the downloaded public asset bundle contents to docs/assets/. Keep capstone_ranked_queue.csv local and untracked.")
try:
    from google.colab import files
    files.download(str(bundle_path))
except Exception:
    print("Not running in Colab; download the ZIP from the working directory.")

## Self-check

- [ ] Hugging Face access and notebook run completed in Colab
- [ ] All three methods compared on the same client-held-out test rows
- [ ] Test base rate shown beside ranking metrics
- [ ] No feature overlaps the future target window
- [ ] IDs used only for grouping / local reconciliation, never as model inputs
- [ ] Error summary and feature importance inspected
- [ ] Public-safe JSON and SVG charts generated and committed under `docs/assets/`
- [ ] Local ranked queue reviewed; no row-level queue committed or published
- [ ] Paper page updated with actual metrics and findings
- [ ] GitHub Pages enabled and verified in a private browser window
- [ ] `submission/paper_url.txt` contains exactly the live paper URL
- [ ] 5-minute demo outline, social-post cut and employer summary prepared

## 10. Five-minute demo, social cut and employer summary

**Demo outline (5 minutes):**
1. (0:00–0:45) State the decision: which visible pages deserve human CTR review first?
2. (0:45–1:30) Explain the two non-overlapping windows and client-content grain.
3. (1:30–2:30) Show the position-adjusted baseline and why position matters.
4. (2:30–3:30) Compare baseline, Logistic Regression and Random Forest on unseen clients.
5. (3:30–4:30) Show the ranked queue, reason codes and error/feature review.
6. (4:30–5:00) Close with limitations: observational data, one holdout, no causal refresh claim.

**Social-post cut (fill the bracketed values after execution):**

I built a CTR opportunity-ranking study on the FlyRank ML Internship warehouse. Using prior search performance to rank pages against a future 30-day under-capture outcome, I compared a position-adjusted rule with Logistic Regression and Random Forest under a client-held-out split. The best model achieved [actual average precision] and [actual Precision@50], versus [baseline values] for the rule. The output is a human-review queue, not a claim about Google's algorithm or guaranteed gains. [Paper URL]

**Employer-facing summary (update after execution):**

I built a reproducible search CTR opportunity-ranking pipeline using DuckDB, pandas and scikit-learn on the FlyRank ML Internship dataset. I engineered leakage-aware prior-window features, compared a transparent baseline with supervised models, and evaluated them on clients held out from training. The final output is a ranked, reason-coded review queue with a public research paper and explicit limits on interpretation.